# Phoneme recognition — Colab training (Model A / Model B, CTC)

Runtime → Change runtime type → **T4 GPU**. (The TPU is not used: CTC loss and variable-length
batches are poorly supported on XLA/TPU.)

**Before the first run** (on your PC): `python scripts/make_code_zip.py`, then upload
`phonrec_code.zip` to **My Drive/phonrec/**.

**If Colab disconnects:** re-run sections 1–4 (they skip finished work), then the training cell —
it resumes from `last.pt` on Drive.

Order: 1 setup → 2 data → 3 features → 4 copy to local disk → 5 train A → 6 inspect → 7 dev eval →
(later) 8 Model B → 9 final test (once).

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU - switch runtime to T4"
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/phonrec')      # everything persistent lives here
REPO  = pathlib.Path('/content/phoneme-recognition')
DATA  = pathlib.Path('/content/data')                        # fast local disk (lost on disconnect)
for p in [DRIVE, DRIVE/'datasets', DRIVE/'runs', DRIVE/'reports', DATA]:
    p.mkdir(parents=True, exist_ok=True)
NPROC = os.cpu_count(); print('CPU cores:', NPROC)

In [ ]:
# code: unzip the uploaded repo (re-run after you upload a new zip)
ZIP = DRIVE/'phonrec_code.zip'
if not ZIP.exists():
    md = pathlib.Path('/content/drive/MyDrive')       # shallow search: a full Drive walk can take minutes
    found = [str(p) for pat in ('phonrec_code*.zip', '*/phonrec_code*.zip') for p in md.glob(pat)]
    raise FileNotFoundError(f'{ZIP} not found. Upload phonrec_code.zip into My Drive/phonrec/ '
                            f'(wait for the upload to finish, then re-run). Found elsewhere: {found or "nothing"}')
!rm -rf {REPO} && unzip -q -o {ZIP} -d /content
assert (REPO/'pyproject.toml').exists(), f'unzip did not produce {REPO}'
%cd {REPO}
!pip -q install -e . cmudict soundfile
!python -m pytest -q -m "not slow" tests 2>&1 | tail -3


In [ ]:
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Data: LibriSpeech (OpenSLR 12, CC BY 4.0) + LibriSpeech lexicon (OpenSLR 11)
Tarballs are cached on Drive (downloaded once, `wget -c` resumes), extracted to local disk each session.
Licence: LibriSpeech is CC BY 4.0 — attribute it in your report. Check the OpenSLR 11 page for the lexicon's terms.

In [ ]:
import subprocess
SUBSETS = ['dev-clean', 'train-clean-100', 'test-clean']
MIRRORS = ['https://www.openslr.org/resources/12', 'https://us.openslr.org/resources/12',
           'https://openslr.elda.org/resources/12']
def fetch(name):
    tgz = DRIVE/'datasets'/f'{name}.tar.gz'
    ok = tgz.with_name(tgz.name + '.done')            # written only after a complete, verified download
    if not ok.exists():
        for m in MIRRORS:                              # wget -c resumes a partial file on any mirror
            r = subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', str(tgz), f'{m}/{name}.tar.gz'])
            if r.returncode == 0 and subprocess.run(['gzip', '-t', str(tgz)]).returncode == 0:
                ok.touch(); break
            print(f'{name}: download from {m} failed or incomplete, trying next mirror')
        else:
            raise RuntimeError(f'could not download {name}; re-run this cell to resume')
    if not (DATA/f'.{name}.extracted').exists():       # marker: an interrupted tar is redone
        subprocess.run(['tar', 'xzf', str(tgz), '-C', str(DATA)], check=True)
        (DATA/f'.{name}.extracted').touch()
    print(name, 'ready')
for s in SUBSETS:
    fetch(s)
lex = DRIVE/'datasets'/'librispeech-lexicon.txt'
if not lex.exists():
    subprocess.run(['wget', '-q', '-O', str(lex), 'https://www.openslr.org/resources/11/librispeech-lexicon.txt'], check=True)
!python scripts/prepare_librispeech.py --root {DATA} --subsets {' '.join(SUBSETS)} --out {DATA}/manifests

## 3. Features (cached on Drive, resumable)
`QUICK=True` builds only the first 6000 training utterances (~20 h) for a fast first experiment.
Set `QUICK=False` later: already-built shards are skipped, the rest is added.
Train = `clean` + `aug1` (random gain + added noise). Wiener filter **on** (bypass A/B is done on your own
recordings, where noise matters).

In [ ]:
QUICK = True
FEATS_DRIVE = DRIVE/'features_v1'
max_utts = '--max-utts 6000' if QUICK else ''
common = f'--out {FEATS_DRIVE} --extra-lexicon {lex} --workers {NPROC} --shard-size 2000 --wiener on'
!python scripts/build_features.py --manifest {DATA}/manifests/dev-clean.csv --split dev --variants clean {common}
!python scripts/build_features.py --manifest {DATA}/manifests/train-clean-100.csv --split train --variants clean aug1 {max_utts} {common}

In [ ]:
import json
for sp in ['train', 'dev']:
    r = json.load(open(FEATS_DRIVE/sp/'phonemize_report.json'))
    print(sp, '| kept', r['kept'], '/', r['rows'], '| unique OOV', r['oov_unique'], '| sources', r['label_sources'])
    print('   top OOV:', r['oov_top'][:15])

## 4. Copy features to local disk (fast random access during training)

In [ ]:
!mkdir -p /content/features && rsync -a --info=progress2 {FEATS_DRIVE}/ /content/features/
!du -sh /content/features/*

## 5. Train Model A
Starting values from the spec: lr 1e-3, weight decay 0.01, clip 5, batch 48, ≤40 epochs, early stop on dev PER.
Checkpoints go to Drive every epoch. Re-running this cell resumes.

In [ ]:
RUN_A = DRIVE/'runs'/'model_a_v1'
!python -m phonrec.training.train --config configs/train_a.json --features /content/features --out {RUN_A}

## 6. Learning curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv(RUN_A/'history.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h.epoch, h.train_loss, label='train'); ax[0].plot(h.epoch, h.dev_loss, label='dev'); ax[0].set_title('CTC loss'); ax[0].legend()
ax[1].plot(h.epoch, 100*h.dev_per); ax[1].set_title('dev PER % (SIL excluded)')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.show(); h.tail()

## 7. Dev evaluation (PER, S/D/I, per-phoneme, confusions, latency); saves log-probs for calibration

In [ ]:
REP_A = DRIVE/'reports'/'model_a_v1_dev'
!python -m phonrec.evaluation.evaluate --ckpt {RUN_A}/best.pt --features /content/features --split dev --out {REP_A} --save-logprobs
r = json.load(open(REP_A/'results.json'))
print('top confusions (ref -> hyp, count):', r['top_confusions'][:12])

## 8. Model B (after Model A works) — lr 5e-4, batch 32

In [ ]:
RUN_B = DRIVE/'runs'/'model_b_v1'
!python -m phonrec.training.train --config configs/train_b.json --features /content/features --out {RUN_B}

## 9. FINAL test-clean evaluation — run once, at the end
Never choose checkpoints, thresholds or the temperature from these numbers.

In [ ]:
!python scripts/build_features.py --manifest {DATA}/manifests/test-clean.csv --split test --variants clean {common}
!rsync -a {FEATS_DRIVE}/test /content/features/
!python -m phonrec.evaluation.evaluate --ckpt {RUN_A}/best.pt --features /content/features --split test --out {DRIVE}/reports/model_a_v1_test

## Getting the model to your PC
`My Drive/phonrec/runs/model_a_v1/best.pt` is self-contained (weights, model config, label map,
front-end config, normaliser). Download it from Drive into `D:\Usman\phoneme-recognition\checkpoints\`.